In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, TensorDataset
import pandas as pd
import numpy as np
import optuna
import gc
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
import matplotlib.pyplot as plt

# Set device for GPU acceleration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Modeling environment ready. Using device: {device}")

🚀 Modeling environment ready. Using device: cuda


In [5]:
# Load the 38-column gold mine
df = pd.read_parquet("delineated_storms.parquet")

# SOTA Feature Selection (Pruned for Multicollinearity)
features = [
    'precip_1hr [inch]', 'precip_max_intensity [inch/hour]', 
    'temp_2m [degF]', 'soil_moisture_05cm [m^3/m^3]', 'elevation [feet]'
]
target = 'depth_inches'

# Downcast to float32 to save 50% RAM
df_model = df[features + [target]].dropna().astype('float32')
print(f"✅ Data Loaded: {len(df_model):,} rows prepared for training.")

✅ Data Loaded: 32,572,701 rows prepared for training.


In [6]:
# 1. Residual Block for ANN
class ResidualBlock(nn.Module):
    def __init__(self, size, dropout_rate=0.2):
        super().__init__()
        self.ln = nn.LayerNorm(size)
        self.fc = nn.Linear(size, size)
        self.dropout = nn.Dropout(dropout_rate)
    def forward(self, x):
        return x + self.dropout(F.relu(self.fc(self.ln(x))))

class SotaANN(nn.Module):
    def __init__(self, input_size, hidden_size=128):
        super().__init__()
        self.input = nn.Linear(input_size, hidden_size)
        self.res1 = ResidualBlock(hidden_size)
        self.res2 = ResidualBlock(hidden_size)
        self.output = nn.Linear(hidden_size, 1)
    def forward(self, x):
        return self.output(self.res2(self.res1(F.relu(self.input(x)))))

# 2. Attention Mechanism for GRU
class AttentionLayer(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.attn = nn.Linear(hidden_size, 1)
    def forward(self, outputs):
        weights = F.softmax(self.attn(outputs), dim=1)
        context = torch.sum(outputs * weights, dim=1)
        return context

class SotaAttentionGRU(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        self.attention = AttentionLayer(hidden_size * 2)
        self.fc = nn.Linear(hidden_size * 2, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(self.attention(out))

In [7]:
class HydroWindowDataset(Dataset):
    def __init__(self, data, targets, window_size):
        self.data = torch.tensor(data, dtype=torch.float32)
        self.targets = torch.tensor(targets, dtype=torch.float32)
        self.window_size = window_size
    def __len__(self):
        return len(self.data) - self.window_size
    def __getitem__(self, idx):
        return self.data[idx : idx + self.window_size], self.targets[idx + self.window_size]

def calculate_nse(y_true, y_pred):
    y_true, y_pred = np.array(y_true).flatten(), np.array(y_pred).flatten()
    return 1 - (np.sum((y_true - y_pred)**2) / np.sum((y_true - np.mean(y_true))**2))

In [8]:
def objective_log_reg(trial):
    alpha = trial.suggest_float("alpha", 1e-3, 10.0, log=True)
    c = trial.suggest_float("log_shift", 1e-4, 0.5, log=True)
    split = int(len(df_model) * 0.8)
    train, test = df_model.iloc[:split], df_model.iloc[split:]
    
    model = Ridge(alpha=alpha).fit(train[features], np.log(train[target] + c))
    preds = np.exp(model.predict(test[features])) - c
    return calculate_nse(test[target].values, preds)

In [9]:
def objective_sota_ann(trial):
    h_size = trial.suggest_int("hidden_size", 64, 512)
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [8192, 16384])

    split = int(len(df_model) * 0.8)
    train_df, test_df = df_model.iloc[:split], df_model.iloc[split:]
    
    scaler_X, scaler_y = StandardScaler(), StandardScaler()
    X_tr = torch.tensor(scaler_X.fit_transform(train_df[features]), dtype=torch.float32)
    y_tr = torch.tensor(scaler_y.fit_transform(train_df[[target]]), dtype=torch.float32)
    
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=batch_size, shuffle=False)
    model = SotaANN(len(features), h_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)

    model.train()
    for _ in range(5):
        for bx, by in loader:
            optimizer.zero_grad()
            torch.nn.MSELoss()(model(bx.to(device)), by.to(device)).backward()
            optimizer.step()
    
    model.eval()
    with torch.no_grad():
        X_te = torch.tensor(scaler_X.transform(test_df[features]), dtype=torch.float32).to(device)
        preds = scaler_y.inverse_transform(model(X_te).cpu().numpy()).flatten()
        return calculate_nse(test_df[target].values, preds)

In [10]:
def objective_attention_gru(trial):
    window_size = trial.suggest_int("window_size", 30, 120, step=30)
    h_size = trial.suggest_int("hidden_size", 32, 128)
    batch_size = 4096 

    split = int(len(df_model) * 0.8)
    train_df, test_df = df_model.iloc[:split], df_model.iloc[split:]
    
    scaler_X, scaler_y = StandardScaler(), StandardScaler()
    X_train_s = scaler_X.fit_transform(train_df[features])
    y_train_s = scaler_y.fit_transform(train_df[[target]])

    train_ds = HydroWindowDataset(X_train_s, y_train_s, window_size)
    loader = DataLoader(train_ds, batch_size=batch_size, shuffle=False)
    
    model = SotaAttentionGRU(len(features), h_size).to(device)
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    
    model.train()
    for _ in range(2): # GRU is complex; 2 epochs on 20M rows is significant
        for bx, by in loader:
            optimizer.zero_grad()
            torch.nn.MSELoss()(model(bx.to(device)), by.to(device)).backward()
            optimizer.step()
            
    # Evaluation Logic (as defined in Cell 4 logic)
    return run_gru_validation(model, test_df, scaler_X, scaler_y, window_size)

In [11]:
# 1. Baseline
study_lr = optuna.create_study(direction="maximize")
study_lr.optimize(objective_log_reg, n_trials=30)

# 2. ANN
study_ann = optuna.create_study(direction="maximize")
study_ann.optimize(objective_sota_ann, n_trials=20)

# 3. GRU (High compute, fewer trials)
study_gru = optuna.create_study(direction="maximize")
study_gru.optimize(objective_attention_gru, n_trials=10)

print(f"🏆 Best Scores:\nLog-Reg: {study_lr.best_value:.4f}\nANN: {study_ann.best_value:.4f}\nGRU: {study_gru.best_value:.4f}")

[I 2026-04-07 16:30:43,474] A new study created in memory with name: no-name-35a81358-6f01-4153-83d0-7226a32d928f
[I 2026-04-07 16:30:45,080] Trial 0 finished with value: -0.026463627815246582 and parameters: {'alpha': 0.0013038490222791784, 'log_shift': 0.00040323319435900347}. Best is trial 0 with value: -0.026463627815246582.
[I 2026-04-07 16:30:46,683] Trial 1 finished with value: -0.02529311180114746 and parameters: {'alpha': 0.006448914068250562, 'log_shift': 0.010385691008497252}. Best is trial 1 with value: -0.02529311180114746.
[I 2026-04-07 16:30:48,286] Trial 2 finished with value: -0.02478158473968506 and parameters: {'alpha': 1.212877887971006, 'log_shift': 0.01621224280730297}. Best is trial 2 with value: -0.02478158473968506.
[I 2026-04-07 16:30:49,888] Trial 3 finished with value: -0.01190173625946045 and parameters: {'alpha': 0.3237107357269315, 'log_shift': 0.36633865640238256}. Best is trial 3 with value: -0.01190173625946045.
[I 2026-04-07 16:30:51,507] Trial 4 fini

OutOfMemoryError: CUDA out of memory. Tried to allocate 11.31 GiB. GPU 0 has a total capacity of 10.57 GiB of which 10.18 GiB is free. Including non-PyTorch memory, this process has 384.00 MiB memory in use. Of the allocated memory 148.24 MiB is allocated by PyTorch, and 37.76 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
# Plotting the results of the best models against ground truth
plt.figure(figsize=(15, 6))
# [Retrieve best predictions logic goes here]
plt.plot(y_obs[-500:], label="Ground Truth", color="black", lw=2)
plt.plot(p_ann[-500:], label="SOTA Residual ANN", color="purple", alpha=0.7)
plt.title("SOTA Urban Flood Prediction: Model Shootout Comparison")
plt.ylabel("Inches")
plt.legend()
plt.show()

NameError: name 'y_obs' is not defined

<Figure size 1500x600 with 0 Axes>